# HF cohort construction

This notebook implements physician-only patient-level annotation,
adjudication, train/test splitting, and five-fold cross-validation.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

RANDOM_SEED = 1
PATIENT_COLUMN = "subject_id"
LABEL_COLUMN = "label_numerical"


## Candidate identification and one-row-per-patient construction


In [ ]:
def normalize_icd(code):
    return str(code).replace(".", "").strip().upper()


def load_code_dictionary(path, code_column="dx code", sheet_name=0):
    dictionary = pd.read_excel(path, sheet_name=sheet_name, dtype=str)
    codes = dictionary[code_column].dropna().map(normalize_icd).unique().tolist()
    if not codes:
        raise ValueError("The diagnosis-code dictionary is empty.")
    return codes


def identify_hf_patients(diagnoses, target_codes):
    required = {"subject_id", "hadm_id", "icd_code"}
    missing = required.difference(diagnoses.columns)
    if missing:
        raise KeyError(f"Diagnosis table is missing columns: {sorted(missing)}")
    normalized_targets = tuple(normalize_icd(code) for code in target_codes)
    table = diagnoses.copy()
    table["normalized_icd_code"] = table["icd_code"].map(normalize_icd)
    table["candidate_code_positive"] = table["normalized_icd_code"].map(
        lambda code: any(code.startswith(target) for target in normalized_targets)
    )
    return table


def one_row_per_patient(candidate_rows, admissions, patients):
    admission_columns = [
        column
        for column in [
            "subject_id",
            "hadm_id",
            "admittime",
            "dischtime",
            "admission_type",
            "race",
        ]
        if column in admissions.columns
    ]
    patient_columns = [
        column
        for column in ["subject_id", "gender", "anchor_age", "anchor_year"]
        if column in patients.columns
    ]
    table = candidate_rows.merge(
        admissions[admission_columns],
        on=["subject_id", "hadm_id"],
        how="left",
        validate="many_to_one",
    )
    table = table.merge(
        patients[patient_columns],
        on="subject_id",
        how="left",
        validate="many_to_one",
    )
    if "admittime" in table.columns:
        table["admittime"] = pd.to_datetime(table["admittime"], errors="coerce")
        table = table.sort_values(["subject_id", "admittime"])
    patient_summary = table.groupby("subject_id", as_index=False).agg(
        candidate_code_positive=("candidate_code_positive", "max"),
        all_icd_codes=(
            "normalized_icd_code",
            lambda values: "|".join(sorted(set(map(str, values)))),
        ),
    )
    first_admission = (
        table.drop(columns=["candidate_code_positive"])
        .groupby("subject_id", as_index=False)
        .first()
    )
    return first_admission.merge(
        patient_summary,
        on="subject_id",
        how="inner",
        validate="one_to_one",
    )


def sample_balanced_candidates(patient_rows, n_per_class=500):
    positives = patient_rows[patient_rows["candidate_code_positive"]].copy()
    negatives = patient_rows[~patient_rows["candidate_code_positive"]].copy()
    if len(positives) < n_per_class or len(negatives) < n_per_class:
        raise ValueError("At least 500 positive and 500 negative candidates are required.")

    positives = positives.sample(n=n_per_class, random_state=RANDOM_SEED)
    unused_negative_indices = set(negatives.index)
    matched_negative_indices = []
    age_column = "anchor_age" if "anchor_age" in patient_rows.columns else None

    for _, positive in positives.sort_values(PATIENT_COLUMN).iterrows():
        candidates = negatives.loc[sorted(unused_negative_indices)].copy()
        score = pd.Series(0.0, index=candidates.index)
        if age_column:
            positive_age = pd.to_numeric(positive.get(age_column), errors="coerce")
            candidate_age = pd.to_numeric(candidates[age_column], errors="coerce")
            score += (candidate_age - positive_age).abs().fillna(20.0)
        for column, mismatch_penalty in (("gender", 20.0), ("race", 5.0)):
            if column in candidates.columns:
                score += (
                    candidates[column].astype(str) != str(positive.get(column))
                ).astype(float) * mismatch_penalty
        selected_index = score.sort_values(kind="stable").index[0]
        matched_negative_indices.append(selected_index)
        unused_negative_indices.remove(selected_index)

    balanced = pd.concat(
        [positives, negatives.loc[matched_negative_indices]],
        ignore_index=True,
    )
    balanced = balanced.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    if balanced[PATIENT_COLUMN].duplicated().any() or len(balanced) != 2 * n_per_class:
        raise AssertionError("Balanced sampling did not produce 1,000 unique patients.")
    return balanced


## Independent physician review and senior adjudication

Two physicians independently assign the binary diagnosis and phenotype.
Agreement is accepted. A senior expert adjudicates diagnostic or phenotype
disagreement. Records are excluded when either independent review is
incomplete, or when the senior expert marks the evidence ambiguous or
insufficient to determine the diagnosis and phenotype.


In [ ]:
VALID_SUBTYPES = {"HFrEF", "HFmrEF", "HFpEF", "Others", "na"}

ANNOTATION_COLUMNS = [
    "physician_1_label",
    "physician_2_label",
    "senior_expert_label",
    "senior_evidence_status",
    "physician_1_subtype",
    "physician_2_subtype",
    "senior_expert_subtype",
]


def create_physician_annotation_sheet(patient_rows):
    if patient_rows[PATIENT_COLUMN].duplicated().any():
        raise ValueError("Annotation must be performed on one row per patient.")
    sheet = patient_rows.copy()
    for column in ANNOTATION_COLUMNS:
        if column not in sheet.columns:
            sheet[column] = pd.NA
    return sheet


def adjudicate_physician_annotations(annotation_sheet):
    required = set(ANNOTATION_COLUMNS).union({PATIENT_COLUMN})
    missing = required.difference(annotation_sheet.columns)
    if missing:
        raise KeyError(f"Annotation sheet is missing columns: {sorted(missing)}")
    if annotation_sheet[PATIENT_COLUMN].duplicated().any():
        raise ValueError("The annotation sheet contains duplicate patient identifiers.")

    def adjudicate_row(row):
        first = pd.to_numeric(row["physician_1_label"], errors="coerce")
        second = pd.to_numeric(row["physician_2_label"], errors="coerce")
        if first not in (0, 1) or second not in (0, 1):
            return pd.Series([False, pd.NA, pd.NA, "incomplete_independent_review"])

        evidence_status = str(row["senior_evidence_status"]).strip().lower()
        senior_label = pd.to_numeric(row["senior_expert_label"], errors="coerce")
        senior_subtype = str(row["senior_expert_subtype"]).strip()

        if first == second:
            label = int(first)
            first_subtype = str(row["physician_1_subtype"]).strip()
            second_subtype = str(row["physician_2_subtype"]).strip()
            if label == 0:
                subtype = "na"
            elif first_subtype == second_subtype and first_subtype in VALID_SUBTYPES - {"na"}:
                subtype = first_subtype
            elif evidence_status == "sufficient" and senior_subtype in VALID_SUBTYPES - {"na"}:
                subtype = senior_subtype
            else:
                return pd.Series(
                    [False, pd.NA, pd.NA, "ambiguous_or_insufficient_phenotype"]
                )
            return pd.Series([True, label, subtype, "physician_agreement"])

        if evidence_status == "sufficient" and senior_label in (0, 1):
            if int(senior_label) == 0:
                senior_subtype = "na"
            elif senior_subtype not in VALID_SUBTYPES - {"na"}:
                return pd.Series(
                    [False, pd.NA, pd.NA, "ambiguous_or_insufficient_phenotype"]
                )
            return pd.Series(
                [True, int(senior_label), senior_subtype, "senior_adjudication"]
            )
        return pd.Series([False, pd.NA, pd.NA, "ambiguous_or_insufficient_evidence"])

    adjudicated = annotation_sheet.copy()
    adjudicated[
        ["include_in_cohort", LABEL_COLUMN, "label_subtype", "adjudication_status"]
    ] = adjudicated.apply(adjudicate_row, axis=1)
    adjudicated = adjudicated[adjudicated["include_in_cohort"]].copy()
    adjudicated[LABEL_COLUMN] = adjudicated[LABEL_COLUMN].astype(int)
    return adjudicated


## Patient-level split and five-fold cross-validation


In [ ]:
EXPECTED_FINAL_PATIENTS = 973
EXPECTED_TRAIN_PATIENTS = 730
EXPECTED_TEST_PATIENTS = 243
DOCUMENTED_EXCLUDED_PATIENTS = 27


def patient_level_train_test_split(adjudicated):
    if adjudicated[PATIENT_COLUMN].duplicated().any():
        raise ValueError("Patient-level splitting requires one row per patient.")
    if len(adjudicated) != EXPECTED_FINAL_PATIENTS:
        raise ValueError(
            f"Expected {EXPECTED_FINAL_PATIENTS} adjudicated patients, found {len(adjudicated)}."
        )
    train, test = train_test_split(
        adjudicated,
        test_size=EXPECTED_TEST_PATIENTS,
        stratify=adjudicated[LABEL_COLUMN],
        random_state=RANDOM_SEED,
    )
    train = train.sort_values(PATIENT_COLUMN).reset_index(drop=True)
    test = test.sort_values(PATIENT_COLUMN).reset_index(drop=True)
    if len(train) != EXPECTED_TRAIN_PATIENTS or len(test) != EXPECTED_TEST_PATIENTS:
        raise AssertionError("The split does not match the documented cohort sizes.")
    if set(train[PATIENT_COLUMN]).intersection(set(test[PATIENT_COLUMN])):
        raise AssertionError("Patient overlap detected between training and test sets.")
    train["split"] = "train"
    test["split"] = "test"
    return train, test


def assign_patient_level_cv_folds(train):
    if train[PATIENT_COLUMN].duplicated().any():
        raise ValueError("Cross-validation requires one row per patient.")
    folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    train = train.copy()
    train["cv_fold"] = -1
    labels = train[LABEL_COLUMN].astype(int).to_numpy()
    groups = train[PATIENT_COLUMN].to_numpy()
    for fold, (_, validation_index) in enumerate(folds.split(train, labels, groups)):
        train.loc[train.index[validation_index], "cv_fold"] = fold
    if set(train["cv_fold"]) != set(range(5)):
        raise AssertionError("Five patient-level folds were not created.")
    return train


def save_cohort_outputs(train, test, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    train.to_csv(output_dir / "hf_train_patient_level.csv", index=False)
    test.to_csv(output_dir / "hf_test_patient_level.csv", index=False)


## Paths and execution


In [ ]:
DATA_ROOT = Path("path/to/mimic")
CODE_DICTIONARY = Path("path/to/hf_diagnosis_codes.xlsx")
ANNOTATION_INPUT = Path("path/to/completed_hf_physician_annotations.csv")
OUTPUT_DIR = Path("processed/hf")


def prepare_annotation_sheet():
    diagnoses = pd.read_csv(DATA_ROOT / "hosp" / "diagnoses_icd.csv", dtype={"icd_code": str})
    admissions = pd.read_csv(DATA_ROOT / "hosp" / "admissions.csv")
    patients = pd.read_csv(DATA_ROOT / "hosp" / "patients.csv")
    target_codes = load_code_dictionary(CODE_DICTIONARY)
    candidates = identify_hf_patients(diagnoses, target_codes)
    patient_rows = one_row_per_patient(candidates, admissions, patients)
    patient_rows = sample_balanced_candidates(patient_rows, n_per_class=500)

    annotation_sheet = create_physician_annotation_sheet(patient_rows)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    annotation_sheet.to_csv(
        OUTPUT_DIR / "hf_physician_annotation_template.csv",
        index=False,
    )
    return annotation_sheet


def finalize_patient_level_cohort():
    completed_annotations = pd.read_csv(ANNOTATION_INPUT)
    adjudicated = adjudicate_physician_annotations(completed_annotations)
    train, test = patient_level_train_test_split(adjudicated)
    train = assign_patient_level_cv_folds(train)
    save_cohort_outputs(train, test, OUTPUT_DIR)
    return train, test


# Run prepare_annotation_sheet, complete physician review, then run finalize_patient_level_cohort.
